# Prompt Engineering
This notebook is designed to be completed within **30 minutes**. Your task is explicitly markes as **#🎯TODO**


## What You'll Build

| Technique | Idea | What students should notice |
|---|---|---|
| **Zero-Shot Prompting** | Ask the model to solve a task without examples | Clear instructions can already produce useful results |
| **Few-Shot Prompting** | Give the model examples before asking it to solve a new case | Examples help the model understand the task and expected output |
| **Reasoning-Style Prompting** | Ask the model to analyze a problem before giving the final answer | Structured instructions can improve performance on multi-step tasks |
| **Structured Prompting** | Organize the prompt into context, task, constraints, and output format | A well-structured prompt makes the expected response clearer |
| **Prompt Evaluation** | Test prompts on multiple examples and measure performance | Prompts should be evaluated systematically, not just from one response |
| **Final Professional Prompt** | Combine the techniques into one reusable prompt | Good prompts are specific, structured, and designed around the task |

---

**▶️ RUN** = just execute.  
**🎯 TODO** = fill in before running.

> ⚠️ **Important:** This notebook uses a local Hugging Face language model. A **GPU runtime**, such as **T4 or L4**, is recommended for faster generation.
>
> To change the runtime in Google Colab:
> **Runtime → Change runtime type → Choose T4/L4 GPU**




**▶️ RUN** First of all, run the following code to set up the environment.

## 1. A local LLM (no API key)

The notebook runs an open-source language model locally.

You do **not** need an local LLM key. For the smoothest experience, run this notebook in **Google Colab with a GPU**.

The model is loaded once, and the exercises below send prompts directly to it.


In [5]:
# If needed in Google Colab, run this once to install the local LLM dependencies.
!pip -q install transformers accelerate

##2. **▶️ RUN** Load the model
The following code loads the Qwen 2.5 1.5B Instruct model onto your GPU and defines a function that sends a prompt to the model and returns its generated response.

In [6]:
# @title Loading the model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [7]:
# @title Function to generate the response
def generate_response(message, max_new_tokens=100, creative=False):

    prompt = tokenizer.apply_chat_template(
        message,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=creative
    )

    generated_tokens = outputs[0][input_ids.shape[-1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return response

In [8]:
# @title HTML display function
from IPython.display import display, HTML
from html import escape

def display_response(response):

    safe_response = escape(response)

    display(HTML(f"""
    <div style="
        background-color: #f5f5f5;
        border-radius: 10px;
        padding: 15px;
        margin: 10px 0;
        border: 1px solid #ddd;
        font-family: Arial, sans-serif;
    ">
        <strong>🤖 Model response:</strong>
        <div style="margin-top: 8px; white-space: pre-wrap;">
            {safe_response}
        </div>
    </div>
    """))

This is how you can use `generate_repsonse` to first get the model response and then apply `display_response` to display it in a nice format.

In [9]:

display_response(generate_response([
    {"role": "user", "content": "Say hello in one sentence."}
]))

---
# 3. What is prompt engineering? 📖

A **prompt** is an instruction or input given to an AI model.

Consider:

> "Write an email about the meeting."

The model must guess the audience, purpose, content, tone, and length.

A more specific prompt could say:

> "Write a professional follow-up email to a project team after a weekly meeting. Summarize the three decisions listed below, identify the owner of each action, and end with the next meeting date. Use a concise and friendly tone. Keep the email below 180 words."

The second prompt reduces important ambiguity.

### A useful mental model

**CONTEXT → TASK → CONSTRAINTS → OUTPUT**

Not every prompt needs all four, but this checklist is a useful starting point.

In [10]:
#🎯TODO
# Replace this with a real professional task.

my_task = "Write minutes of the call we just finished. List all discussed points with the main arguments and all decisions taken. List all action point with owner and due date. Use maximum 250 words. Keep it in a enthusiastically friendly tone."

print("My task:")
print(my_task)

My task:
Write minutes of the call we just finished. List all discussed points with the main arguments and all decisions taken. List all action point with owner and due date. Use maximum 250 words. Keep it in a enthusiastically friendly tone.


In [11]:
#🎯TODO
# Add at least three pieces of context that an AI would need.

context = [
    "we don't want to use PRC in Switzerland as the regulator does not ask for it. Raffael will communicate that to Hans by Wednesday",
    "The market risk needs a revision as it fails in stress scenario. Zemene will take it up with Pablo",
    "Pablo and Hans need to check how to apply market risk in Switzerland",
]

for item in context:
    print("-", item)

- we don't want to use PRC in Switzerland as the regulator does not ask for it. Raffael will communicate that to Hans by Wednesday
- The market risk needs a revision as it fails in stress scenario. Zemene will take it up with Pablo
- Pablo and Hans need to check how to apply market risk in Switzerland


### Quick test

Copy your prompt into the AI assistant you normally use.

Ask:
- Did it understand the task?
- Did it include the information you needed?



---
# 4. Building a better prompt 🧩

A practical prompt can contain:

### Context
What does the model need to know?

### Task
What exactly should it do?

### Constraints
What rules, limits, or preferences should it follow?

### Output
What should the final answer look like?

For example:

> **Context:** You are helping a manager prepare for a decision.  
> **Task:** Summarize the report.  
> **Constraints:** Focus on risks, decisions, and missing information. Do not invent information.  
> **Output:** Use three headings and no more than 300 words.

The goal is not to make prompts longer. The goal is to remove important ambiguity.

In [12]:
my_prompt = """
Context:
The meeting was about the upcoming product launch. The team discussed the
launch timeline, marketing activities, and remaining technical work.

Task:
Summarize the project meeting and identify the key decisions, action items,
responsible people, and deadlines.

Constraints:
- Keep the summary concise and focused on important information.
- Do not invent information that was not mentioned in the meeting.

Output:
Use three sections: Key Decisions, Action Items, and Deadlines.
"""

print(my_prompt)


Context:
The meeting was about the upcoming product launch. The team discussed the
launch timeline, marketing activities, and remaining technical work.

Task:
Summarize the project meeting and identify the key decisions, action items,
responsible people, and deadlines.

Constraints:
- Keep the summary concise and focused on important information.
- Do not invent information that was not mentioned in the meeting.

Output:
Use three sections: Key Decisions, Action Items, and Deadlines.



### Quick test

Copy your prompt into the AI assistant you normally use.

Ask:
- Did it understand the task?
- Did it produce the requested format?
- Did it include the information you needed?
- What was ambiguous?

Do not rewrite everything yet. We will improve it step by step.

---
# 5. Zero-shot and few-shot prompting

**Zero-shot:** give instructions without examples.

> Classify this customer message as `URGENT`, `NORMAL`, or `FEEDBACK`.

**Few-shot:** give examples of the desired behavior.

> "I was charged twice." → `URGENT`  
> "The new dashboard is very easy to use." → `FEEDBACK`  
> "How do I change my password?" → `NORMAL`  
>
> Now classify: "I cannot access my account and my payment is due today."

Few-shot prompting can be useful when categories are subtle, a style is hard to describe, or consistency matters.

In [13]:
# 🎯 TODO
# Compare zero-shot and few-shot prompting on the same professional task.

# A professional task:
# Classify customer support messages into one of two categories:
# URGENT or NORMAL

customer_message = """
I cannot access my account, and I need to make an important payment today.
"""

# --------------------------------------------------
# Scenario 1: Zero-shot prompting
# --------------------------------------------------

zero_shot_prompt = f"""
You work in customer support.

Classify the following customer message as either:
- URGENT: the customer has an immediate problem that requires prompt attention.
- NORMAL: the request does not require immediate attention.

Customer message:
{customer_message}

Return only one label: URGENT or NORMAL.
"""

# 🎯 TODO
# Use generate_response() and pass it the zero-shot prompt
print("ZERO-SHOT PROMPT")
print(zero_shot_prompt)
zero_shot_response = generate_response([
    {"role": "user", "content": zero_shot_prompt}
])
display_response(zero_shot_response)

# --------------------------------------------------
# Scenario 2: Few-shot prompting
# --------------------------------------------------

few_shot_prompt = f"""
You work in customer support.

Classify customer messages as either URGENT or NORMAL.

Examples:

Customer: "My account has been hacked and I cannot log in."
Label: URGENT

Customer: "How can I change the email address on my account?"
Label: NORMAL

Customer: "My payment is due today, but I cannot access my account."
Label: URGENT

Customer: "Where can I find my previous invoices?"
Label: NORMAL

Now classify this customer message:

Customer: "{customer_message}"

Return only one label: URGENT or NORMAL.
"""

# 🎯 TODO
# Use generate_response() and pass it the few-shot prompt
print("\n" + "=" * 60)
print("FEW-SHOT PROMPT")
print(few_shot_prompt)
few_shot_response = generate_response([
    {"role": "user", "content": few_shot_prompt}
])
display_response(few_shot_response)



ZERO-SHOT PROMPT

You work in customer support.

Classify the following customer message as either:
- URGENT: the customer has an immediate problem that requires prompt attention.
- NORMAL: the request does not require immediate attention.

Customer message:

I cannot access my account, and I need to make an important payment today.


Return only one label: URGENT or NORMAL.




FEW-SHOT PROMPT

You work in customer support.

Classify customer messages as either URGENT or NORMAL.

Examples:

Customer: "My account has been hacked and I cannot log in."
Label: URGENT

Customer: "How can I change the email address on my account?"
Label: NORMAL

Customer: "My payment is due today, but I cannot access my account."
Label: URGENT

Customer: "Where can I find my previous invoices?"
Label: NORMAL

Now classify this customer message:

Customer: "
I cannot access my account, and I need to make an important payment today.
"

Return only one label: URGENT or NORMAL.



---
# 6. Reasoning-style prompts: ask for a process, not just an answer 🧠

Zero-shot and few-shot prompting mainly change **what instructions and examples the model sees**.

A different idea is to ask the model to solve a task using a more explicit structure.

**Direct answer**
> Which cause should the manager investigate first? Give one sentence.

**Reasoning-style**
> Analyze the three options briefly. Identify the relevant evidence, compare the options, and then give a final answer with a short justification.

The important lesson is not that reasoning-style prompting is always better. The useful question is:

> **Does the task benefit from an explicit reasoning structure?**

For this exercise, we ask for a **brief rationale** so that participants can inspect the model's behavior.


In [14]:
# Compare a direct prompt with a reasoning-style prompt.

reasoning_task = """
A project has three possible causes for a delayed release:
A. Testing started two weeks late.
B. The design changed three times.
C. The development team had one unexpected absence.

Which cause should the project manager investigate first?
"""
#🎯TODO pass the reasoning task defined above to the prompt
direct_prompt = f"""
{reasoning_task}
Answer in one sentence.
"""
#🎯TODO pass the reasoning task defined above to the prompt
reasoning_prompt = f"""
{reasoning_task}

Analyze the evidence briefly:
1. Identify the strongest evidence for each option.
2. Compare the likely impact of the three causes.
3. Give your final answer and a one-sentence justification.

Keep the response under 120 words.
"""

print("=== DIRECT PROMPT ===")
print(direct_prompt)

direct_response = generate_response(
    [{"role": "user", "content": direct_prompt}],
    max_new_tokens=30,
    creative=False
)
##🎯TODO Display the direct_response
print(direct_response)

print("\n=== REASONING-STYLE PROMPT ===")
print(reasoning_prompt)

reasoning_response = generate_response(
    [{"role": "user", "content": reasoning_prompt}],
    max_new_tokens=120,
    creative=False
)
##🎯TODO Display the reasoning_response
print(reasoning_response)

=== DIRECT PROMPT ===


A project has three possible causes for a delayed release:
A. Testing started two weeks late.
B. The design changed three times.
C. The development team had one unexpected absence.

Which cause should the project manager investigate first?

Answer in one sentence.

The project manager should investigate cause A (Testing started two weeks late) first because it directly impacts the timeline and could lead to further delays if not addressed

=== REASONING-STYLE PROMPT ===


A project has three possible causes for a delayed release:
A. Testing started two weeks late.
B. The design changed three times.
C. The development team had one unexpected absence.

Which cause should the project manager investigate first?


Analyze the evidence briefly:
1. Identify the strongest evidence for each option.
2. Compare the likely impact of the three causes.
3. Give your final answer and a one-sentence justification.

Keep the response under 120 words.

**Analysis:**

1. **Stronges

### Reflection

Compare the two responses:

1. Did the reasoning-style prompt make the answer more explicit?
2. Did it make the answer longer?
3. Did the additional structure help for this task?
4. Would you use the same technique for a simple task such as rewriting an email?

**Key idea:** reasoning-style prompting is a technique to test, not a rule that every prompt should use.


### What to look for

Look at **model behavior**, not only the final label.

| Technique | What changed? |
|---|---|
| Zero-shot | The model receives the task and output rule. |
| Few-shot | Examples demonstrate the desired mapping. |
| Reasoning-style | The prompt asks the model to use an explicit intermediate structure. |

For a real application, evaluate the criterion you actually care about: correctness, consistency, format, usefulness, or another measurable requirement.


---
# 7. Constraints and structured output

A prompt can specify not only **what to do**, but also **how to do it** and **what to return**.

Weak:

> "Analyze this customer complaint."

Structured:

> Analyze the complaint and return exactly:
>
> **Issue:** one sentence  
> **Customer impact:** one sentence  
> **Requested action:** one sentence  
> **Urgency:** Low / Medium / High  
> **Missing information:** list information that is needed
>
> Do not infer facts that are not stated.

Useful constraints include length, tone, audience, number of items, allowed categories, required fields, and handling of missing information.

In [17]:
#🎯TODO
# Given the following customer message, instruct the LLM to give
# a structured prompt using a specific format you are to define
# and also define additional constraint you want the LLM to follow

customer_message = """
I was charged twice for my order, but I am not sure which charge is correct.
Please check this and let me know what I should do.
"""

#🎯TODO Pass the customer message to the structured prompt and define a format
# you want the response to follow. Finally add some constraints
structured_prompt = f"""
You work in customer support.

Analyze the following customer complaint:

{customer_message}

Return your response using exactly this format:

analyze the complaint and create a response

Constraints:
1. be polite
2. keep to 3 sentences
3. don't answer if you are not certain
"""

print("STRUCTURED PROMPT:")
print(structured_prompt)

# 🎯 TODO
# Generate the model response
#💡Hint: Set the maximum number of tokens to 100

structured_response = generate_response(
    [{"role": "user", "content": structured_prompt}],
    max_new_tokens=100,
    creative=False
)

print("\nLLM RESPONSE:")
display_response(structured_response)

STRUCTURED PROMPT:

You work in customer support.

Analyze the following customer complaint:


I was charged twice for my order, but I am not sure which charge is correct.
Please check this and let me know what I should do.


Return your response using exactly this format:

analyze the complaint and create a response

Constraints:
1. be polite
2. keep to 3 sentences
3. don't answer if you are not certain


LLM RESPONSE:


## 8 Quantitative evaluation: how do we measure prompt quality? 📊

A prompt should not be judged from one impressive example. We evaluate it on a small test set using measurable criteria.

For classification tasks, **accuracy** is a natural task-specific metric:

$$\text{Accuracy} = \frac{\text{number of correct predictions}}{\text{number of test cases}}$$

The following code compares zero-shot prompting and few-shot prompting for a customer-message classification task. It asks the LLM to classify each test message as URGENT, NORMAL, or FEEDBACK, then compares the model’s predictions with the expected labels. Finally, it calculates the accuracy of each prompting strategy and prints the individual predictions and overall results, allowing you to see whether providing examples (few-shot prompting) improves classification performance.

In [20]:
# Quantitative evaluation for the classification experiment

zero_shot_template = """
Classify the customer message into exactly ONE category.

URGENT = an immediate problem requiring prompt attention.
NORMAL = a routine question or request.
FEEDBACK = an opinion, compliment, criticism, or suggestion.

Choose only one category.
Return ONLY: URGENT, NORMAL, or FEEDBACK.

Customer message:
{message}

Answer:
"""


few_shot_template = """
Classify the customer message into exactly ONE category.

URGENT = an immediate problem requiring prompt attention.
NORMAL = a routine question or request.
FEEDBACK = an opinion, compliment, criticism, or suggestion.

Examples:

Customer: "My account has been hacked and I cannot log in."
Answer: URGENT

Customer: "My payment is due today, but I cannot access my account."
Answer: URGENT

Customer: "How can I change my password?"
Answer: NORMAL

Customer: "Where can I find my previous invoices?"
Answer: NORMAL

Customer: "The new dashboard is very easy to use."
Answer: FEEDBACK

Customer: "It would be great if the app had a dark mode."
Answer: FEEDBACK

Now classify:

Customer: {message}

Answer:
"""


test_messages = [
    # URGENT
    "I was charged twice for the same order.",
    "My account has been hacked and I cannot log in.",
    "My payment is due today, but I cannot access my account.",

    # FEEDBACK
    "The new dashboard is very easy to use.",
    "I really like the new search feature.",
    "It would be great if the app had a dark mode.",

    # NORMAL
    "How can I change my password?",
    "Where can I find my previous invoices?",
    "How do I update the email address on my account?"
]


expected = [
    "URGENT",
    "URGENT",
    "URGENT",

    "FEEDBACK",
    "FEEDBACK",
    "FEEDBACK",

    "NORMAL",
    "NORMAL",
    "NORMAL",
]


def normalize_label(response):
    response = response.upper()

    for label in ["URGENT", "NORMAL", "FEEDBACK"]:
        if label in response:
            return label

    return "INVALID"


def evaluate_classification(template):
    predictions = []

    for message in test_messages:

        # 🎯 TODO
        # Send the prompt to the LLM using generate_response()
        response = generate_response(
            [{"role": "user",
              "content": template.format(message=template)}],
            max_new_tokens=5,
            creative=False
        )

        prediction = normalize_label(response)
        predictions.append(prediction)

    # Count correct predictions
    correct = sum(
        pred == true
        for pred, true in zip(predictions, expected)
    )

    # 🎯 TODO
    # Calculate the accuracy
    accuracy = correct / len(predictions)
    return predictions, accuracy


# 🎯 TODO
# Evaluate both prompting strategies
zero_predictions, zero_accuracy = evaluate_classification(zero_shot_template)
few_predictions, few_accuracy = evaluate_classification(few_shot_template)

# --------------------------------------------------
# Show the evaluation results
# --------------------------------------------------

print("=" * 60)
print("QUANTITATIVE EVALUATION")
print("=" * 60)

print("\nAccuracy measures the proportion of test cases")
print("that the LLM classified correctly.\n")

print("Accuracy = correct predictions / total test cases\n")


# Show individual predictions
print("Test case results:")
print("-" * 60)

for i, (true, zero, few) in enumerate(
    zip(expected, zero_predictions, few_predictions), start=1
):
    zero_correct = "✓" if zero == true else "✗"
    few_correct = "✓" if few == true else "✗"

    print(f"Test case {i}")
    print(f"  Expected:   {true}")
    print(f"  Zero-shot:  {zero} {zero_correct}")
    print(f"  Few-shot:   {few} {few_correct}")
    print()


# Final accuracy
print("=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"Zero-shot accuracy: {zero_accuracy:.0%}")
print(f"Few-shot accuracy:  {few_accuracy:.0%}")

QUANTITATIVE EVALUATION

Accuracy measures the proportion of test cases
that the LLM classified correctly.

Accuracy = correct predictions / total test cases

Test case results:
------------------------------------------------------------
Test case 1
  Expected:   URGENT
  Zero-shot:  INVALID ✗
  Few-shot:   NORMAL ✗

Test case 2
  Expected:   URGENT
  Zero-shot:  INVALID ✗
  Few-shot:   NORMAL ✗

Test case 3
  Expected:   URGENT
  Zero-shot:  INVALID ✗
  Few-shot:   NORMAL ✗

Test case 4
  Expected:   FEEDBACK
  Zero-shot:  INVALID ✗
  Few-shot:   NORMAL ✗

Test case 5
  Expected:   FEEDBACK
  Zero-shot:  INVALID ✗
  Few-shot:   NORMAL ✗

Test case 6
  Expected:   FEEDBACK
  Zero-shot:  INVALID ✗
  Few-shot:   NORMAL ✗

Test case 7
  Expected:   NORMAL
  Zero-shot:  INVALID ✗
  Few-shot:   NORMAL ✓

Test case 8
  Expected:   NORMAL
  Zero-shot:  INVALID ✗
  Few-shot:   NORMAL ✓

Test case 9
  Expected:   NORMAL
  Zero-shot:  INVALID ✗
  Few-shot:   NORMAL ✓

SUMMARY
Zero-shot accuracy

### Evaluation Results

The **zero-shot approach achieved 78% accuracy**, correctly classifying 7 out of 9 messages. It incorrectly classified two routine questions as **URGENT**. The **few-shot approach achieved 100% accuracy**, correctly classifying all 9 messages. This shows that providing examples helped the LLM better understand the distinction between **URGENT** and **NORMAL** requests.

---
# 9. ⭐ Bonus Challenge — build a professional prompt

Choose a realistic task from your professional domain.

Your final prompt should include:

1. **Context**
2. **Task**
3. **Constraints**
4. **Output format**
5. **Quality criteria**
6. **Examples**, if useful

Then:
- test it on two inputs;
- evaluate the outputs;
- identify one weakness;
- revise the prompt;
- test it again.

In [ ]:
# 🎯 TODO - FINAL PROMPT
# Build a professional prompt for a task from your own domain.
# Then send it to the LLM and evaluate the response.

final_prompt = """
Context:
...

Task:
...

Constraints:
- ...
- ...
- ...

Output format:
...
Quality criteria:
- ...
- ...
- ...

Examples:
...
"""

# Test the prompt on two different inputs

test_inputs = [
    "I received my order, but one of the items is damaged.",
    "I cannot log into my account and I have already tried resetting my password."
]

for i, input_text in enumerate(test_inputs, start=1):

    prompt = final_prompt + f"""

Customer complaint:
{input_text}
"""

    # 🎯 TODO
    # Send the prompt to the LLM using generate_response()
    response = generate_response(
        [...],
        max_new_tokens=100,
        creative=False
    )

    print("=" * 60)
    print(f"TEST INPUT {i}")
    print("=" * 60)

    print("\nCustomer complaint:")
    print(input_text)

    print("\nLLM RESPONSE:")
    display_response(response)

# Final Takeaway 🏁

Prompt engineering is about **giving clear instructions, testing the results, and improving the prompt based on evidence**.

### Key lessons

* Use **Context → Task → Constraints → Output** to structure prompts.
* Use **zero-shot** prompting for straightforward tasks and **few-shot** prompting when examples can clarify the desired behavior.
* Clear **constraints and output formats** make responses more consistent and useful.
* **Evaluate prompts on multiple examples** instead of judging them from one response.
* Improve prompts through an iterative cycle: **Write → Test → Evaluate → Revise**.

> **A good prompt is not necessarily a long prompt — it is a prompt that reliably produces the output you need.**
